# Nemotron Nano: End-to-End Model Workflow

This notebook walks through the complete lifecycle of working with a large language model
using the Agentic Assistants framework, from downloading weights off HuggingFace Hub to
fine-tuning, evaluation, and building agentic applications.

## Contents

1. **Setup and Configuration** -- Dependencies, adapters, Ollama
2. **HuggingFace Hub** -- Authentication, model/dataset discovery, download, contributing
3. **Fine-Tuning with Unsloth** -- 4-bit QLoRA training with MLflow tracking
4. **Evaluation and Experiment Tracking** -- Metrics, MLflow comparison
5. **Agentic Paper Extraction (LangGraph)** -- Multi-step extraction of data models, process flows, and abstractions
6. **Deep Agents** -- Autonomous research analysis with planning and subagents

## Prerequisites

```bash
# Core + training + fine-tuning extras
poetry install -E fine-tuning -E nemotron -E assistant

# Ollama must be running locally
ollama serve
```

**Hardware**: A CUDA-capable GPU with >=16 GB VRAM is recommended for fine-tuning.
CPU-only mode works for the HuggingFace and agentic sections.

In [1]:
import subprocess, importlib, sys

def _ensure(package: str, pip_name: str | None = None):
    try:
        importlib.import_module(package)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", pip_name or package]
        )

# _ensure("unsloth")
_ensure("deepagents")
_ensure("datasets")
_ensure("peft")
_ensure("trl")

print("Optional dependencies verified.")

W0321 22:21:46.321000 3264 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Optional dependencies verified.


In [2]:
import os 
os.environ['HF_TOKEN'] = ''

In [3]:
import os

from agentic_assistants import AgenticConfig, OllamaManager
from agentic_assistants.integrations.huggingface import (
    HuggingFaceHubIntegration,
    ModelCard,
    DatasetCard,
)
from agentic_assistants.adapters import LangGraphAdapter
from agentic_assistants.core import MLFlowTracker
from agentic_assistants.utils.logging import setup_logging

setup_logging(level="INFO")

config = AgenticConfig()
tracker = MLFlowTracker(config)
adapter = LangGraphAdapter(config)

HF_TOKEN = os.environ.get("HF_TOKEN", None)
hf = HuggingFaceHubIntegration(token=HF_TOKEN)

print(f"MLflow enabled : {tracker.enabled}")
print(f"Tracking URI   : {config.mlflow.tracking_uri}")
print(f"HF Hub ready   : {hf.is_available}")
print(f"Storage backend: {hf.get_storage_backend()}")

[03/21/26 22:21:58] INFO     OpenTelemetry initialized - endpoint: http://localhost:4317, service: telemetry.py:124
                             agentic-assistants                                                                    

MLflow enabled : True
Tracking URI   : http://localhost:5000
HF Hub ready   : True
Storage backend: huggingface_hub


In [4]:
ollama = OllamaManager(config)
ollama.ensure_running()
model_name = ollama.ensure_model(model_name='nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8')
print(f"Ollama model ready: {model_name}")

[03/21/26 22:22:04] INFO     Model nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8 not found, pulling...    ollama.py:383

                    INFO     Pulling model: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8                 ollama.py:304

                    INFO     Successfully pulled model: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8     ollama.py:325

Ollama model ready: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8


---

## Part 2 -- HuggingFace Hub

This section demonstrates the full round-trip with HuggingFace Hub:
authenticating, discovering models and datasets, downloading weights,
loading datasets, and contributing back.

### 2.1 Authentication

Set `HF_TOKEN` in your environment (or a `.env` file) before running this notebook.
Gated models like Nemotron Nano require an accepted license agreement on the Hub.

In [5]:
user_info = hf.whoami()
if user_info:
    print(f"Logged in as: {user_info['name']}")
    print(f"Orgs: {[o['name'] for o in user_info.get('orgs', [])]}")
else:
    print(
        "Not authenticated -- set HF_TOKEN to enable gated-model access "
        "and push operations.  Read-only public access still works."
    )

Logged in as: julesthecomputernerd
Orgs: ['algovision']


### 2.2 Discover Models

In [6]:
nemotron_models = hf.search_models(
    query="nemotron",
    author="nvidia",
    task="text-generation",
    sort="downloads",
    limit=10,
)

for m in nemotron_models:
    print(f"  {m['id']:60s}  downloads={m['downloads']:>10,}")

BASE_MODEL_ID = "nvidia/Llama-3.1-Nemotron-Nano-8B-v1"
info = hf.get_model_info(BASE_MODEL_ID)
if info:
    print(f"\nTarget model: {info['id']}")
    print(f"  Downloads : {info['downloads']:,}")
    print(f"  Likes     : {info['likes']:,}")
    print(f"  Tags      : {info['tags'][:8]}")

  nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-FP8                     downloads= 1,490,501
  nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16                    downloads=   962,410
  nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4                downloads=   629,612
  nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8                  downloads=   586,472
  nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-NVFP4                   downloads=   571,528
  nvidia/NVIDIA-Nemotron-Nano-9B-v2-Japanese                    downloads=   423,267
  nvidia/NVIDIA-Nemotron-Nano-9B-v2                             downloads=   358,294
  nvidia/Llama-3.1-Nemotron-Nano-8B-v1                          downloads=   294,449
  nvidia/NVIDIA-Nemotron-Nano-9B-v2-Base                        downloads=   158,351
  nvidia/Llama-3_3-Nemotron-Super-49B-v1_5                      downloads=   112,624

Target model: nvidia/Llama-3.1-Nemotron-Nano-8B-v1
  Downloads : 294,449
  Likes     : 221
  Tags      : ['transformers', 'safetensors', 'llama', 'te

### 2.3 Download Model Weights

This uses `pull_model` which calls `snapshot_download` under the hood.
For a single file (e.g. just the config), use `download_file` instead.

In [7]:
model_path = hf.pull_model(BASE_MODEL_ID)
print(f"Model downloaded to: {model_path}")

config_path = hf.download_file(BASE_MODEL_ID, "config.json")
if config_path:
    import json
    with open(config_path) as f:
        model_config = json.load(f)
    print(f"\nModel architecture : {model_config.get('model_type')}")
    print(f"Hidden size        : {model_config.get('hidden_size')}")
    print(f"Num attention heads: {model_config.get('num_attention_heads')}")
    print(f"Num layers         : {model_config.get('num_hidden_layers')}")

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

[03/21/26 22:24:01] INFO     Downloaded model to C:\Users\Julian                                 huggingface.py:348
                             Wiley\Documents\GitHub\agentic_assistants\notebooks\data\models\cac                   
                             he\nvidia_Llama-3.1-Nemotron-Nano-8B-v1                                               

Model downloaded to: C:\Users\Julian Wiley\Documents\GitHub\agentic_assistants\notebooks\data\models\cache\nvidia_Llama-3.1-Nemotron-Nano-8B-v1


                    INFO     Downloaded config.json from nvidia/Llama-3.1-Nemotron-Nano-8B-v1   huggingface.py:1269
                             to                                                                                    
                             data\models\cache\files\nvidia_Llama-3.1-Nemotron-Nano-8B-v1\confi                    
                             g.json                                                                                


Model architecture : llama
Hidden size        : 4096
Num attention heads: 32
Num layers         : 32


### 2.4 Load Datasets

In [ ]:
DATASET_ID = "nvidia/HelpSteer2"

ds_info = hf.get_dataset_info(DATASET_ID)
if ds_info:
    print(f"Dataset: {ds_info['id']}")
    print(f"  Downloads: {ds_info['downloads']:,}")

dataset = hf.load_hf_dataset(DATASET_ID, split="train", streaming=True)

print("\nFirst 3 examples (streaming):")
for i, example in enumerate(dataset):
    if i >= 3:
        break
    prompt_preview = str(example.get("prompt", ""))[:120]
    print(f"  [{i}] {prompt_preview}...")

### 2.5 Contributing Back to the Hub

After fine-tuning you can push adapter weights, updated model cards,
and curated datasets back to HuggingFace Hub.

In [ ]:
card = hf.create_model_card(
    model_name="my-nemotron-nano-finetuned",
    base_model=BASE_MODEL_ID,
    description="Nemotron Nano fine-tuned with QLoRA on a custom instruction dataset.",
    training_method="qlora",
    training_dataset=DATASET_ID,
    metrics={"eval_loss": 0.85, "perplexity": 6.2},
    tags=["nemotron", "qlora", "instruction-tuning"],
)
print("Generated model card preview:")
print(card.to_markdown()[:600])
print("...")

# Uncomment to push (requires write access):
# result = hf.push_model(
#     model_path="./data/models/my-nemotron-nano-finetuned",
#     repo_id="your-username/my-nemotron-nano-finetuned",
#     model_card=card,
#     private=True,
# )
# print(result)

### 2.6 Search Papers

In [ ]:
papers = hf.search_papers("nemotron", limit=5)
for p in papers:
    title = p.get("title", p.get("id", "unknown"))
    print(f"  - {title}")

---

## Part 3 -- Fine-Tuning with Unsloth + MLflow

Unsloth provides optimised LoRA kernels that significantly reduce VRAM usage
and speed up training compared to stock PEFT/TRL. Below we:

1. Load the base model in 4-bit (QLoRA) via Unsloth
2. Apply LoRA adapters
3. Prepare an instruction-tuning dataset
4. Fine-tune with TRL's `SFTTrainer` and log to MLflow
5. Export and optionally push the adapter

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto-detect
)

print(f"Model loaded: {type(model).__name__}")
print(f"Tokenizer vocab size: {tokenizer.vocab_size:,}")
print(f"Device: {model.device}")

In [ ]:
from agentic_assistants.training.config import LoRAConfig

lora_cfg = LoRAConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_cfg.r,
    lora_alpha=lora_cfg.lora_alpha,
    lora_dropout=lora_cfg.lora_dropout,
    target_modules=lora_cfg.target_modules,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
from datasets import load_dataset

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def format_alpaca(examples):
    texts = []
    for instruction, inp, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = ALPACA_PROMPT.format(
            instruction=instruction, input=inp, output=output
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

raw_dataset = load_dataset("yahma/alpaca-cleaned", split="train")
train_dataset = raw_dataset.map(format_alpaca, batched=True)

print(f"Training samples: {len(train_dataset):,}")
print(f"Example:\n{train_dataset[0]['text'][:300]}...")

In [ ]:
from trl import SFTTrainer, SFTConfig
import mlflow

OUTPUT_DIR = "./data/models/nemotron-nano-qlora"

mlflow.set_tracking_uri(config.mlflow.tracking_uri)
mlflow.set_experiment("nemotron-nano-finetuning")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    fp16=not (torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False) and torch.cuda.is_available(),
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,
    report_to=["mlflow"],
    run_name="nemotron-qlora-unsloth",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

print("Trainer ready. Starting fine-tuning...")
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Loss         : {train_result.training_loss:.4f}")
print(f"  Global steps : {train_result.global_step}")
print(f"  Runtime (s)  : {train_result.metrics.get('train_runtime', 0):.1f}")

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

# Optionally push adapter weights to Hub
# result = hf.push_weights(
#     weights_path=OUTPUT_DIR,
#     repo_id="your-username/nemotron-nano-qlora",
#     adapter_name="default",
#     private=True,
# )
# print(result)

---

## Part 4 -- Evaluation and Experiment Tracking

After training we evaluate the fine-tuned model, log metrics to MLflow,
and compare experiment runs.

In [ ]:
import math

FastLanguageModel.for_inference(model)

eval_prompts = [
    "Explain the difference between LoRA and QLoRA in two sentences.",
    "Write a Python function that computes the Fibonacci sequence using memoization.",
    "What are the key benefits of using MLflow for experiment tracking?",
]

print("=" * 70)
for prompt_text in eval_prompts:
    inputs = tokenizer(
        ALPACA_PROMPT.format(instruction=prompt_text, input="", output=""),
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = decoded.split("### Response:")[-1].strip()
    print(f"Q: {prompt_text}")
    print(f"A: {response[:400]}")
    print("-" * 70)

In [ ]:
eval_dataset = load_dataset("yahma/alpaca-cleaned", split="train").select(range(100))
eval_dataset = eval_dataset.map(format_alpaca, batched=True)

total_loss = 0.0
num_batches = 0

for i in range(0, len(eval_dataset), 8):
    batch_texts = eval_dataset[i : i + 8]["text"]
    encodings = tokenizer(
        batch_texts, return_tensors="pt", padding=True, truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(model.device)
    with torch.no_grad():
        out = model(**encodings, labels=encodings["input_ids"])
    total_loss += out.loss.item()
    num_batches += 1

avg_loss = total_loss / max(num_batches, 1)
perplexity = math.exp(avg_loss)

print(f"Evaluation loss : {avg_loss:.4f}")
print(f"Perplexity      : {perplexity:.2f}")

In [ ]:
with tracker.start_run(run_name="nemotron-eval") as run:
    tracker.log_param("base_model", BASE_MODEL_ID)
    tracker.log_param("method", "qlora")
    tracker.log_param("lora_r", lora_cfg.r)
    tracker.log_param("lora_alpha", lora_cfg.lora_alpha)
    tracker.log_param("dataset", "yahma/alpaca-cleaned")
    tracker.log_metric("eval_loss", avg_loss)
    tracker.log_metric("perplexity", perplexity)
    tracker.log_metric("trainable_params", trainable)
    print(f"Logged evaluation to MLflow run: {run.info.run_id}")

In [ ]:
runs_df = mlflow.search_runs(
    experiment_names=["nemotron-nano-finetuning"],
    order_by=["start_time DESC"],
    max_results=10,
)

display_cols = [
    col for col in runs_df.columns
    if col.startswith("params.") or col.startswith("metrics.") or col == "run_id"
]
if not runs_df.empty:
    print(runs_df[display_cols[:12]].to_string(index=False))
else:
    print("No runs found yet -- run the training cell above first.")

---

## Part 5 -- Agentic Paper Extraction with LangGraph

Build a multi-step agent that reads a research paper and extracts:

- **Data models** -- structured schemas and entities described in the paper
- **Process flows** -- algorithms, pipelines, and step-by-step procedures
- **Abstractions** -- architectural patterns, design decisions, and key concepts

The graph uses the `LangGraphAdapter` from the framework for observability.

In [8]:
from typing import TypedDict, Annotated, Sequence, List
from operator import add
from pydantic import BaseModel, Field


class DataModelSchema(BaseModel):
    name: str = Field(description="Name of the data model or entity")
    fields: List[str] = Field(description="Fields/attributes of the model")
    description: str = Field(description="What this data model represents")


class ProcessFlow(BaseModel):
    name: str = Field(description="Name of the process or algorithm")
    steps: List[str] = Field(description="Ordered steps in the process")
    description: str = Field(description="Purpose of the process flow")


class Abstraction(BaseModel):
    name: str = Field(description="Name of the abstraction or pattern")
    category: str = Field(description="Category: architectural, algorithmic, conceptual")
    description: str = Field(description="Explanation of the abstraction")


class PaperAnalysisState(TypedDict):
    paper_query: str
    raw_text: str
    data_models: str
    process_flows: str
    abstractions: str
    synthesis: str
    steps: Annotated[Sequence[str], add]


print("State and output schemas defined.")

State and output schemas defined.


In [9]:
llm = adapter.create_ollama_llm()


def retrieve_paper(state: PaperAnalysisState) -> PaperAnalysisState:
    """Fetch paper content via the HuggingFace papers API."""
    query = state["paper_query"]
    papers = hf.search_papers(query, limit=3)

    if papers:
        paper_texts = []
        for p in papers:
            title = p.get("title", p.get("id", "untitled"))
            summary = p.get("summary", p.get("abstract", "No abstract available."))
            paper_texts.append(f"Title: {title}\nAbstract: {summary}")
        raw = "\n\n---\n\n".join(paper_texts)
    else:
        raw = (
            f"No papers found for '{query}'. Using the query as a topic description "
            "for the extraction nodes to work with."
        )

    return {"raw_text": raw, "steps": ["paper_retrieved"]}


def extract_data_models(state: PaperAnalysisState) -> PaperAnalysisState:
    """Extract data models / entities described in the paper."""
    prompt = (
        "You are an expert software architect. Given the following research paper "
        "content, identify and describe all data models, schemas, and structured "
        "entities. For each, provide: name, key fields/attributes, and a short "
        "description.\n\n"
        f"Paper content:\n{state['raw_text'][:4000]}\n\n"
        "List the data models in a structured format."
    )
    response = llm.invoke(prompt)
    return {"data_models": response.content, "steps": ["data_models_extracted"]}


def extract_process_flows(state: PaperAnalysisState) -> PaperAnalysisState:
    """Extract algorithms, pipelines, and step-by-step procedures."""
    prompt = (
        "You are an expert systems analyst. Given the following research paper "
        "content, identify all process flows, algorithms, and pipelines described. "
        "For each, provide: name, ordered list of steps, and purpose.\n\n"
        f"Paper content:\n{state['raw_text'][:4000]}\n\n"
        "List each process flow with its steps."
    )
    response = llm.invoke(prompt)
    return {"process_flows": response.content, "steps": ["process_flows_extracted"]}


def extract_abstractions(state: PaperAnalysisState) -> PaperAnalysisState:
    """Extract key abstractions, patterns, and design decisions."""
    prompt = (
        "You are an expert researcher. Given the following research paper content, "
        "identify the key abstractions, architectural patterns, and design decisions. "
        "Categorize each as: architectural, algorithmic, or conceptual. Provide a "
        "brief explanation for each.\n\n"
        f"Paper content:\n{state['raw_text'][:4000]}\n\n"
        "List all abstractions found."
    )
    response = llm.invoke(prompt)
    return {"abstractions": response.content, "steps": ["abstractions_extracted"]}


def synthesize(state: PaperAnalysisState) -> PaperAnalysisState:
    """Combine all extracted information into a coherent summary."""
    prompt = (
        "Synthesize the following analysis of a research paper into a concise, "
        "structured summary.\n\n"
        f"DATA MODELS:\n{state['data_models'][:2000]}\n\n"
        f"PROCESS FLOWS:\n{state['process_flows'][:2000]}\n\n"
        f"ABSTRACTIONS:\n{state['abstractions'][:2000]}\n\n"
        "Provide a unified summary that connects the data models, process flows, "
        "and abstractions into a coherent whole."
    )
    response = llm.invoke(prompt)
    return {"synthesis": response.content, "steps": ["synthesis_complete"]}


print("Node functions defined.")

Node functions defined.


In [10]:
from langgraph.graph import StateGraph, END

graph = adapter.create_state_graph(PaperAnalysisState)

graph.add_node("retrieve", adapter.wrap_node(retrieve_paper, "retrieve"))
graph.add_node("data_models", adapter.wrap_node(extract_data_models, "data_models"))
graph.add_node("process_flows", adapter.wrap_node(extract_process_flows, "process_flows"))
graph.add_node("abstractions", adapter.wrap_node(extract_abstractions, "abstractions"))
graph.add_node("synthesize", adapter.wrap_node(synthesize, "synthesize"))

graph.set_entry_point("retrieve")
graph.add_edge("retrieve", "data_models")
graph.add_edge("data_models", "process_flows")
graph.add_edge("process_flows", "abstractions")
graph.add_edge("abstractions", "synthesize")
graph.add_edge("synthesize", END)

workflow = graph.compile()
print("Paper analysis workflow compiled.")

Paper analysis workflow compiled.


In [11]:
initial_state: PaperAnalysisState = {
    "paper_query": "Nemotron language model training optimization",
    "raw_text": "",
    "data_models": "",
    "process_flows": "",
    "abstractions": "",
    "synthesis": "",
    "steps": [],
}

result = adapter.run_graph(
    workflow,
    inputs=initial_state,
    experiment_name="paper-extraction",
    run_name="nemotron-paper-analysis",
)

print("Steps completed:", result["steps"])
print("\n" + "=" * 70)
print("DATA MODELS:\n")
print(result["data_models"][:1500])
print("\n" + "=" * 70)
print("PROCESS FLOWS:\n")
print(result["process_flows"][:1500])
print("\n" + "=" * 70)
print("ABSTRACTIONS:\n")
print(result["abstractions"][:1500])
print("\n" + "=" * 70)
print("SYNTHESIS:\n")
print(result["synthesis"][:2000])

[03/21/26 22:29:30] INFO     Using MinIO artifact storage: s3://agentic-artifacts/ (endpoint: mlflow_tracker.py:151
                             http://minio.data-services.svc.cluster.local:9000)                                    

[03/21/26 22:29:33] INFO     MLFlow initialized - tracking URI: http://localhost:5000,        mlflow_tracker.py:163
                             experiment: paper-extraction                                                          

                    INFO       Artifact store: MinIO/S3                                       mlflow_tracker.py:170

[03/21/26 22:29:34] INFO     Starting LangGraph workflow: nemotron-paper-analysis          langgraph_adapter.py:141

[03/21/26 22:29:38] ERROR    Exception while exporting Span.                                        __init__.py:192
                             ╭──────────────── Traceback (most recent call last) ─────────────────╮                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\sdk\_shared_intern │                
                             │ al\__init__.py:179 in _export                                      │                
                             │                                                                    │                
                             │   176 │   │   │   │   iteration += 1                               │                
                             │   177 │   │   │   │   token = attach(set_value(_SUPPRESS_INSTRUMEN │                
                             │   178 │   │   │   │   try:                                         │                
                             │ ❱ 179 │   │   │   │   │   self._exporter.export(                   │                
                             │   180 │   │   │   │   │   │   [                                    │                
                             │   181 │   │   │   │   │   │   │   # Oldest records are at the back │                
                             │   182 │   │   │   │   │   │   │   self._queue.pop()                │                
                             │                                                                    │                
                             │ ╭──────────────────────────── locals ────────────────────────────╮ │                
                             │ │ batch_strategy = <BatchExportStrategy.EXPORT_AT_LEAST_ONE_BAT… │ │                
                             │ │                  2>                                            │ │                
                             │ │      iteration = 1                                             │ │                
                             │ │           self = <opentelemetry.sdk._shared_internal.BatchPro… │ │                
                             │ │                  object at 0x0000024CD33C09D0>                 │ │                
                             │ │          token = <Token var=<ContextVar name='current_context' │ │                
                             │ │                  default={} at 0x0000024CDE0DF3D0> at          │ │                
                             │ │                  0x0000024D409953C0>                           │ │                
                             │ ╰────────────────────────────────────────────────────────────────╯ │                
                             │                                                                    │                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\exporter\otlp\prot │                
                             │ o\grpc\trace_exporter\__init__.py:146 in export                    │                
                             │                                                                    │                
                             │   143 │   │   return encode_spans(data)                            │                
                             │   144 │                                                            │                
                             │   145 │   def export(self, spans: Sequence[ReadableSpan]) -> SpanE │                
                             │ ❱ 146 │   │   return self

[03/21/26 22:30:14] ERROR    Exception while exporting Span.                                        __init__.py:192
                             ╭──────────────── Traceback (most recent call last) ─────────────────╮                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\sdk\_shared_intern │                
                             │ al\__init__.py:179 in _export                                      │                
                             │                                                                    │                
                             │   176 │   │   │   │   iteration += 1                               │                
                             │   177 │   │   │   │   token = attach(set_value(_SUPPRESS_INSTRUMEN │                
                             │   178 │   │   │   │   try:                                         │                
                             │ ❱ 179 │   │   │   │   │   self._exporter.export(                   │                
                             │   180 │   │   │   │   │   │   [                                    │                
                             │   181 │   │   │   │   │   │   │   # Oldest records are at the back │                
                             │   182 │   │   │   │   │   │   │   self._queue.pop()                │                
                             │                                                                    │                
                             │ ╭──────────────────────────── locals ────────────────────────────╮ │                
                             │ │ batch_strategy = <BatchExportStrategy.EXPORT_AT_LEAST_ONE_BAT… │ │                
                             │ │                  2>                                            │ │                
                             │ │      iteration = 1                                             │ │                
                             │ │           self = <opentelemetry.sdk._shared_internal.BatchPro… │ │                
                             │ │                  object at 0x0000024CD33C09D0>                 │ │                
                             │ │          token = <Token var=<ContextVar name='current_context' │ │                
                             │ │                  default={} at 0x0000024CDE0DF3D0> at          │ │                
                             │ │                  0x0000024D409953C0>                           │ │                
                             │ ╰────────────────────────────────────────────────────────────────╯ │                
                             │                                                                    │                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\exporter\otlp\prot │                
                             │ o\grpc\trace_exporter\__init__.py:146 in export                    │                
                             │                                                                    │                
                             │   143 │   │   return encode_spans(data)                            │                
                             │   144 │                                                            │                
                             │   145 │   def export(self, spans: Sequence[ReadableSpan]) -> SpanE │                
                             │ ❱ 146 │   │   return self

[03/21/26 22:30:24] ERROR    Exception while exporting Span.                                        __init__.py:192
                             ╭──────────────── Traceback (most recent call last) ─────────────────╮                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\sdk\_shared_intern │                
                             │ al\__init__.py:179 in _export                                      │                
                             │                                                                    │                
                             │   176 │   │   │   │   iteration += 1                               │                
                             │   177 │   │   │   │   token = attach(set_value(_SUPPRESS_INSTRUMEN │                
                             │   178 │   │   │   │   try:                                         │                
                             │ ❱ 179 │   │   │   │   │   self._exporter.export(                   │                
                             │   180 │   │   │   │   │   │   [                                    │                
                             │   181 │   │   │   │   │   │   │   # Oldest records are at the back │                
                             │   182 │   │   │   │   │   │   │   self._queue.pop()                │                
                             │                                                                    │                
                             │ ╭──────────────────────────── locals ────────────────────────────╮ │                
                             │ │ batch_strategy = <BatchExportStrategy.EXPORT_AT_LEAST_ONE_BAT… │ │                
                             │ │                  2>                                            │ │                
                             │ │      iteration = 1                                             │ │                
                             │ │           self = <opentelemetry.sdk._shared_internal.BatchPro… │ │                
                             │ │                  object at 0x0000024CD33C09D0>                 │ │                
                             │ │          token = <Token var=<ContextVar name='current_context' │ │                
                             │ │                  default={} at 0x0000024CDE0DF3D0> at          │ │                
                             │ │                  0x0000024D409953C0>                           │ │                
                             │ ╰────────────────────────────────────────────────────────────────╯ │                
                             │                                                                    │                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\exporter\otlp\prot │                
                             │ o\grpc\trace_exporter\__init__.py:146 in export                    │                
                             │                                                                    │                
                             │   143 │   │   return encode_spans(data)                            │                
                             │   144 │                                                            │                
                             │   145 │   def export(self, spans: Sequence[ReadableSpan]) -> SpanE │                
                             │ ❱ 146 │   │   return self

[03/21/26 22:30:31] WARNING  Failed to log dict artifact output/final_state.json: No module   mlflow_tracker.py:344
                             named 'boto3'                                                                         

                    INFO     Graph completed in 57.21s                                     langgraph_adapter.py:159

🏃 View run nemotron-paper-analysis at: http://localhost:5000/#/experiments/696575613720232672/runs/2c84573429db4e60a459616db61c0b98
🧪 View experiment at: http://localhost:5000/#/experiments/696575613720232672
Steps completed: ['paper_retrieved', 'data_models_extracted', 'process_flows_extracted', 'abstractions_extracted', 'synthesis_complete']

DATA MODELS:

Here are the data models, schemas, and structured entities identified from the research paper content:

1. **VEGA-3D Framework**
	* Key fields/attributes:
		+ Pre-trained video diffusion model
		+ Latent World Simulator
		+ Token-level adaptive gated fusion mechanism
	* Description: A plug-and-play framework that repurposes a pre-trained video diffusion model as a Latent World Simulator for generating rich geometric cues without explicit 3D supervision.

2. **Matryoshka Gaussian Splatting (MGS) Framework**
	* Key fields/attributes:
		+ Standard 3D Gaussian Splatting pipeline
		+ Single ordered set of Gaussians
		+ Stochastic budget

In [12]:
print("Streaming extraction workflow:")
print("-" * 40)

initial_state["paper_query"] = "reinforcement learning from human feedback"
initial_state["steps"] = []

for step_output in adapter.stream_graph(
    workflow,
    inputs=initial_state,
    experiment_name="paper-extraction-streaming",
):
    for node_name, node_output in step_output.items():
        steps = node_output.get("steps", [])
        print(f"  Node: {node_name:20s}  steps={steps}")

Streaming extraction workflow:
----------------------------------------
  Node: retrieve              steps=['paper_retrieved']


[03/21/26 22:36:05] ERROR    Exception while exporting Span.                                        __init__.py:192
                             ╭──────────────── Traceback (most recent call last) ─────────────────╮                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\sdk\_shared_intern │                
                             │ al\__init__.py:179 in _export                                      │                
                             │                                                                    │                
                             │   176 │   │   │   │   iteration += 1                               │                
                             │   177 │   │   │   │   token = attach(set_value(_SUPPRESS_INSTRUMEN │                
                             │   178 │   │   │   │   try:                                         │                
                             │ ❱ 179 │   │   │   │   │   self._exporter.export(                   │                
                             │   180 │   │   │   │   │   │   [                                    │                
                             │   181 │   │   │   │   │   │   │   # Oldest records are at the back │                
                             │   182 │   │   │   │   │   │   │   self._queue.pop()                │                
                             │                                                                    │                
                             │ ╭──────────────────────────── locals ────────────────────────────╮ │                
                             │ │ batch_strategy = <BatchExportStrategy.EXPORT_AT_LEAST_ONE_BAT… │ │                
                             │ │                  2>                                            │ │                
                             │ │      iteration = 1                                             │ │                
                             │ │           self = <opentelemetry.sdk._shared_internal.BatchPro… │ │                
                             │ │                  object at 0x0000024CD33C09D0>                 │ │                
                             │ │          token = <Token var=<ContextVar name='current_context' │ │                
                             │ │                  default={} at 0x0000024CDE0DF3D0> at          │ │                
                             │ │                  0x0000024D40C0CC00>                           │ │                
                             │ ╰────────────────────────────────────────────────────────────────╯ │                
                             │                                                                    │                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\exporter\otlp\prot │                
                             │ o\grpc\trace_exporter\__init__.py:146 in export                    │                
                             │                                                                    │                
                             │   143 │   │   return encode_spans(data)                            │                
                             │   144 │                                                            │                
                             │   145 │   def export(self, spans: Sequence[ReadableSpan]) -> SpanE │                
                             │ ❱ 146 │   │   return self

  Node: data_models           steps=['data_models_extracted']
  Node: process_flows         steps=['process_flows_extracted']


[03/21/26 22:36:20] ERROR    Exception while exporting Span.                                        __init__.py:192
                             ╭──────────────── Traceback (most recent call last) ─────────────────╮                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\sdk\_shared_intern │                
                             │ al\__init__.py:179 in _export                                      │                
                             │                                                                    │                
                             │   176 │   │   │   │   iteration += 1                               │                
                             │   177 │   │   │   │   token = attach(set_value(_SUPPRESS_INSTRUMEN │                
                             │   178 │   │   │   │   try:                                         │                
                             │ ❱ 179 │   │   │   │   │   self._exporter.export(                   │                
                             │   180 │   │   │   │   │   │   [                                    │                
                             │   181 │   │   │   │   │   │   │   # Oldest records are at the back │                
                             │   182 │   │   │   │   │   │   │   self._queue.pop()                │                
                             │                                                                    │                
                             │ ╭──────────────────────────── locals ────────────────────────────╮ │                
                             │ │ batch_strategy = <BatchExportStrategy.EXPORT_AT_LEAST_ONE_BAT… │ │                
                             │ │                  2>                                            │ │                
                             │ │      iteration = 1                                             │ │                
                             │ │           self = <opentelemetry.sdk._shared_internal.BatchPro… │ │                
                             │ │                  object at 0x0000024CD33C09D0>                 │ │                
                             │ │          token = <Token var=<ContextVar name='current_context' │ │                
                             │ │                  default={} at 0x0000024CDE0DF3D0> at          │ │                
                             │ │                  0x0000024D40B59F00>                           │ │                
                             │ ╰────────────────────────────────────────────────────────────────╯ │                
                             │                                                                    │                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\exporter\otlp\prot │                
                             │ o\grpc\trace_exporter\__init__.py:146 in export                    │                
                             │                                                                    │                
                             │   143 │   │   return encode_spans(data)                            │                
                             │   144 │                                                            │                
                             │   145 │   def export(self, spans: Sequence[ReadableSpan]) -> SpanE │                
                             │ ❱ 146 │   │   return self

  Node: abstractions          steps=['abstractions_extracted']
  Node: synthesize            steps=['synthesis_complete']
🏃 View run graph-stream-20260321-223558 at: http://localhost:5000/#/experiments/696575613720232672/runs/d2c8dcc48b874ad2a190630b9b8a48b4
🧪 View experiment at: http://localhost:5000/#/experiments/696575613720232672


[03/21/26 22:36:41] ERROR    Exception while exporting Span.                                        __init__.py:192
                             ╭──────────────── Traceback (most recent call last) ─────────────────╮                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\sdk\_shared_intern │                
                             │ al\__init__.py:179 in _export                                      │                
                             │                                                                    │                
                             │   176 │   │   │   │   iteration += 1                               │                
                             │   177 │   │   │   │   token = attach(set_value(_SUPPRESS_INSTRUMEN │                
                             │   178 │   │   │   │   try:                                         │                
                             │ ❱ 179 │   │   │   │   │   self._exporter.export(                   │                
                             │   180 │   │   │   │   │   │   [                                    │                
                             │   181 │   │   │   │   │   │   │   # Oldest records are at the back │                
                             │   182 │   │   │   │   │   │   │   self._queue.pop()                │                
                             │                                                                    │                
                             │ ╭──────────────────────────── locals ────────────────────────────╮ │                
                             │ │ batch_strategy = <BatchExportStrategy.EXPORT_AT_LEAST_ONE_BAT… │ │                
                             │ │                  2>                                            │ │                
                             │ │      iteration = 1                                             │ │                
                             │ │           self = <opentelemetry.sdk._shared_internal.BatchPro… │ │                
                             │ │                  object at 0x0000024CD33C09D0>                 │ │                
                             │ │          token = <Token var=<ContextVar name='current_context' │ │                
                             │ │                  default={} at 0x0000024CDE0DF3D0> at          │ │                
                             │ │                  0x0000024D40B59F00>                           │ │                
                             │ ╰────────────────────────────────────────────────────────────────╯ │                
                             │                                                                    │                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\exporter\otlp\prot │                
                             │ o\grpc\trace_exporter\__init__.py:146 in export                    │                
                             │                                                                    │                
                             │   143 │   │   return encode_spans(data)                            │                
                             │   144 │                                                            │                
                             │   145 │   def export(self, spans: Sequence[ReadableSpan]) -> SpanE │                
                             │ ❱ 146 │   │   return self

---

## Part 6 -- Deep Agents: Autonomous Research Analysis

LangChain's **Deep Agents** library provides a structured runtime for
autonomous multi-step agents with built-in planning, file management,
and subagent delegation. Below we set up a Deep Agent that autonomously
analyses a research paper, plans its extraction strategy, and writes
structured outputs.

Deep Agents is built on LangGraph and uses the standard tool-calling
loop with additional built-in tools for planning (`write_todos`),
file I/O, and subagent spawning (`task`).

In [13]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool


@tool
def search_hf_papers(query: str) -> str:
    """Search HuggingFace Hub for research papers matching the query."""
    papers = hf.search_papers(query, limit=5)
    if not papers:
        return f"No papers found for: {query}"
    lines = []
    for p in papers:
        title = p.get("title", p.get("id", "unknown"))
        abstract = p.get("summary", p.get("abstract", "N/A"))[:300]
        lines.append(f"Title: {title}\nAbstract: {abstract}")
    return "\n---\n".join(lines)


@tool
def search_hf_models(query: str, author: str = "") -> str:
    """Search HuggingFace Hub for models matching the query."""
    models = hf.search_models(query=query, author=author or None, limit=5)
    if not models:
        return f"No models found for: {query}"
    lines = []
    for m in models:
        lines.append(f"{m['id']}  (downloads: {m['downloads']:,})")
    return "\n".join(lines)


custom_tools = [search_hf_papers, search_hf_models]
print(f"Custom tools defined: {[t.name for t in custom_tools]}")

Custom tools defined: ['search_hf_papers', 'search_hf_models']


In [16]:
try:
    from deepagents import create_agent

    llm_for_agent = ChatOllama(
        model=model_name,
        base_url=config.ollama.host,
    )

    agent = create_agent(
        model=llm_for_agent,
        tools=custom_tools,
    )

    print("Deep Agent created with custom HuggingFace tools.")
    print(f"  Model   : {model_name}")
    print(f"  Tools   : {[t.name for t in custom_tools]}")

except Exception as e:
    print(
        "deepagents not installed. Install with:\n",
        "  pip install deepagents\n",
        "Falling back to a LangGraph-based equivalent below.",
        f"{e}"
    )

deepagents not installed. Install with:
   pip install deepagents
 Falling back to a LangGraph-based equivalent below. cannot import name 'create_agent' from 'deepagents' (C:\Users\Julian Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants-iq0iFedS-py3.11\Lib\site-packages\deepagents\__init__.py)


In [15]:
!pip install deepagents


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
ANALYSIS_PROMPT = """Analyse a research paper on the topic of "large language model alignment \
and reinforcement learning from human feedback".

Follow these steps:
1. Search for relevant papers using the search_hf_papers tool.
2. From the results, extract:
   a. Data models -- structured entities, schemas, and data representations
   b. Process flows -- algorithms, training pipelines, step-by-step procedures
   c. Key abstractions -- architectural patterns, design decisions, conceptual frameworks
3. Provide a structured summary connecting all three categories.

Be thorough and specific. Cite the paper titles in your analysis."""

if agent is not None:
    response = agent.invoke({"messages": [{"role": "user", "content": ANALYSIS_PROMPT}]})

    final_message = response["messages"][-1]
    print("Deep Agent Analysis")
    print("=" * 70)
    print(final_message.content)
else:
    print("Deep Agent not available -- using the LangGraph workflow from Part 5.")
    fallback_state: PaperAnalysisState = {
        "paper_query": "large language model alignment RLHF",
        "raw_text": "",
        "data_models": "",
        "process_flows": "",
        "abstractions": "",
        "synthesis": "",
        "steps": [],
    }
    fallback_result = adapter.run_graph(
        workflow,
        inputs=fallback_state,
        experiment_name="deep-agent-fallback",
        run_name="langgraph-paper-analysis-fallback",
    )
    print("Fallback Analysis")
    print("=" * 70)
    print(fallback_result["synthesis"][:2000])

Deep Agent not available -- using the LangGraph workflow from Part 5.


[03/21/26 22:40:03] INFO     Starting LangGraph workflow:                                  langgraph_adapter.py:141
                             langgraph-paper-analysis-fallback                                                     

[03/21/26 22:40:06] ERROR    Exception while exporting Span.                                        __init__.py:192
                             ╭──────────────── Traceback (most recent call last) ─────────────────╮                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\sdk\_shared_intern │                
                             │ al\__init__.py:179 in _export                                      │                
                             │                                                                    │                
                             │   176 │   │   │   │   iteration += 1                               │                
                             │   177 │   │   │   │   token = attach(set_value(_SUPPRESS_INSTRUMEN │                
                             │   178 │   │   │   │   try:                                         │                
                             │ ❱ 179 │   │   │   │   │   self._exporter.export(                   │                
                             │   180 │   │   │   │   │   │   [                                    │                
                             │   181 │   │   │   │   │   │   │   # Oldest records are at the back │                
                             │   182 │   │   │   │   │   │   │   self._queue.pop()                │                
                             │                                                                    │                
                             │ ╭──────────────────────────── locals ────────────────────────────╮ │                
                             │ │ batch_strategy = <BatchExportStrategy.EXPORT_AT_LEAST_ONE_BAT… │ │                
                             │ │                  2>                                            │ │                
                             │ │      iteration = 1                                             │ │                
                             │ │           self = <opentelemetry.sdk._shared_internal.BatchPro… │ │                
                             │ │                  object at 0x0000024CD33C09D0>                 │ │                
                             │ │          token = <Token var=<ContextVar name='current_context' │ │                
                             │ │                  default={} at 0x0000024CDE0DF3D0> at          │ │                
                             │ │                  0x0000024D40BB61C0>                           │ │                
                             │ ╰────────────────────────────────────────────────────────────────╯ │                
                             │                                                                    │                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\exporter\otlp\prot │                
                             │ o\grpc\trace_exporter\__init__.py:146 in export                    │                
                             │                                                                    │                
                             │   143 │   │   return encode_spans(data)                            │                
                             │   144 │                                                            │                
                             │   145 │   def export(self, spans: Sequence[ReadableSpan]) -> SpanE │                
                             │ ❱ 146 │   │   return self

[03/21/26 22:40:22] ERROR    Exception while exporting Span.                                        __init__.py:192
                             ╭──────────────── Traceback (most recent call last) ─────────────────╮                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\sdk\_shared_intern │                
                             │ al\__init__.py:179 in _export                                      │                
                             │                                                                    │                
                             │   176 │   │   │   │   iteration += 1                               │                
                             │   177 │   │   │   │   token = attach(set_value(_SUPPRESS_INSTRUMEN │                
                             │   178 │   │   │   │   try:                                         │                
                             │ ❱ 179 │   │   │   │   │   self._exporter.export(                   │                
                             │   180 │   │   │   │   │   │   [                                    │                
                             │   181 │   │   │   │   │   │   │   # Oldest records are at the back │                
                             │   182 │   │   │   │   │   │   │   self._queue.pop()                │                
                             │                                                                    │                
                             │ ╭──────────────────────────── locals ────────────────────────────╮ │                
                             │ │ batch_strategy = <BatchExportStrategy.EXPORT_AT_LEAST_ONE_BAT… │ │                
                             │ │                  2>                                            │ │                
                             │ │      iteration = 1                                             │ │                
                             │ │           self = <opentelemetry.sdk._shared_internal.BatchPro… │ │                
                             │ │                  object at 0x0000024CD33C09D0>                 │ │                
                             │ │          token = <Token var=<ContextVar name='current_context' │ │                
                             │ │                  default={} at 0x0000024CDE0DF3D0> at          │ │                
                             │ │                  0x0000024D40BB61C0>                           │ │                
                             │ ╰────────────────────────────────────────────────────────────────╯ │                
                             │                                                                    │                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\exporter\otlp\prot │                
                             │ o\grpc\trace_exporter\__init__.py:146 in export                    │                
                             │                                                                    │                
                             │   143 │   │   return encode_spans(data)                            │                
                             │   144 │                                                            │                
                             │   145 │   def export(self, spans: Sequence[ReadableSpan]) -> SpanE │                
                             │ ❱ 146 │   │   return self

[03/21/26 22:40:41] WARNING  Failed to log dict artifact output/final_state.json: No module   mlflow_tracker.py:344
                             named 'boto3'                                                                         

                    INFO     Graph completed in 38.11s                                     langgraph_adapter.py:159

🏃 View run langgraph-paper-analysis-fallback at: http://localhost:5000/#/experiments/696575613720232672/runs/dc15e41d4e2e4033890c348b8304b6af
🧪 View experiment at: http://localhost:5000/#/experiments/696575613720232672
Fallback Analysis
The research paper presents a set of innovative approaches to generative models in 3D scene understanding, spatial reasoning, and embodied manipulation. The following summary synthesizes the key findings from the analysis:

**Data Models:**

The paper introduces five data models, schemas, and structured entities that enable novel approaches to generative modeling:

1. **VEGA-3D Framework**: A plug-and-play framework that repurposes pre-trained video diffusion models as Latent World Simulators, enriching Multimodal Large Language Models (MLLMs) with dense geometric cues.
2. **Matryoshka Gaussian Splatting (MGS) Framework**: A training framework enabling continuous Level of Detail (LoD) for standard 3D Gaussian Splatting pipelines without sacrificing full

[03/21/26 22:40:42] ERROR    Exception while exporting Span.                                        __init__.py:192
                             ╭──────────────── Traceback (most recent call last) ─────────────────╮                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\sdk\_shared_intern │                
                             │ al\__init__.py:179 in _export                                      │                
                             │                                                                    │                
                             │   176 │   │   │   │   iteration += 1                               │                
                             │   177 │   │   │   │   token = attach(set_value(_SUPPRESS_INSTRUMEN │                
                             │   178 │   │   │   │   try:                                         │                
                             │ ❱ 179 │   │   │   │   │   self._exporter.export(                   │                
                             │   180 │   │   │   │   │   │   [                                    │                
                             │   181 │   │   │   │   │   │   │   # Oldest records are at the back │                
                             │   182 │   │   │   │   │   │   │   self._queue.pop()                │                
                             │                                                                    │                
                             │ ╭──────────────────────────── locals ────────────────────────────╮ │                
                             │ │ batch_strategy = <BatchExportStrategy.EXPORT_AT_LEAST_ONE_BAT… │ │                
                             │ │                  2>                                            │ │                
                             │ │      iteration = 1                                             │ │                
                             │ │           self = <opentelemetry.sdk._shared_internal.BatchPro… │ │                
                             │ │                  object at 0x0000024CD33C09D0>                 │ │                
                             │ │          token = <Token var=<ContextVar name='current_context' │ │                
                             │ │                  default={} at 0x0000024CDE0DF3D0> at          │ │                
                             │ │                  0x0000024D40C26A00>                           │ │                
                             │ ╰────────────────────────────────────────────────────────────────╯ │                
                             │                                                                    │                
                             │ C:\Users\Julian                                                    │                
                             │ Wiley\AppData\Local\pypoetry\Cache\virtualenvs\agentic-assistants- │                
                             │ iq0iFedS-py3.11\Lib\site-packages\opentelemetry\exporter\otlp\prot │                
                             │ o\grpc\trace_exporter\__init__.py:146 in export                    │                
                             │                                                                    │                
                             │   143 │   │   return encode_spans(data)                            │                
                             │   144 │                                                            │                
                             │   145 │   def export(self, spans: Sequence[ReadableSpan]) -> SpanE │                
                             │ ❱ 146 │   │   return self

In [18]:
print("\n" + "=" * 70)
print("Comparison: LangGraph vs Deep Agents")
print("=" * 70)
print("""
LangGraph Approach (Part 5):
  - Explicit state graph with defined nodes and edges
  - Full control over execution order and data flow
  - Deterministic: same graph, same execution path
  - Integrated MLflow tracking via LangGraphAdapter
  - Best for: well-defined, repeatable extraction pipelines

Deep Agents Approach (Part 6):
  - Autonomous planning via write_todos
  - Dynamic tool selection and subagent delegation
  - Non-deterministic: agent decides its own strategy
  - Built-in file I/O and context management
  - Best for: open-ended research and exploratory analysis

Both approaches can be combined: use LangGraph for the structured
backbone and Deep Agents for autonomous sub-tasks within nodes.
""")


Comparison: LangGraph vs Deep Agents

LangGraph Approach (Part 5):
  - Explicit state graph with defined nodes and edges
  - Full control over execution order and data flow
  - Deterministic: same graph, same execution path
  - Integrated MLflow tracking via LangGraphAdapter
  - Best for: well-defined, repeatable extraction pipelines

Deep Agents Approach (Part 6):
  - Autonomous planning via write_todos
  - Dynamic tool selection and subagent delegation
  - Non-deterministic: agent decides its own strategy
  - Built-in file I/O and context management
  - Best for: open-ended research and exploratory analysis

Both approaches can be combined: use LangGraph for the structured
backbone and Deep Agents for autonomous sub-tasks within nodes.

